# EA1 – Carga de Datos

**Objetivo:** cargar datos desde Hugging Face, convertirlos a un DataFrame, realizar cuatro transformaciones distintas y generar un resumen en un archivo `.txt`.

El flujo usa dos clases: `HFDatasetLoader` (carga) y `DatasetSummarizer` (resumen y exportación). Ejecuta las celdas en orden.

## 1. Instalar dependencias

In [27]:
%pip -q install datasets pandas

## 2. Crear los archivos Python en Colab
Esta celda crea los dos archivos `.py` para que las importaciones funcionen dentro de Colab.

#Carga un dataset público de Hugging Face y lo convierte en DataFrame

In [28]:
%%writefile hf_dataset_loader.py
"""Carga un dataset público de Hugging Face y lo convierte en DataFrame."""

import pandas as pd
from datasets import load_dataset


class HFDatasetLoader:
    """Extrae un split de Hugging Face y lo devuelve como DataFrame de pandas."""

    def __init__(self, dataset_id: str, split: str = "train"):
        self.dataset_id = dataset_id
        self.split = split

    def load_dataframe(self) -> pd.DataFrame:
        """Carga el dataset indicado y devuelve sus datos como DataFrame."""
        dataset = load_dataset(self.dataset_id, split=self.split)
        return dataset.to_pandas()


Overwriting hf_dataset_loader.py


#Resume un DataFrame y exporta el resumen a un archivo TXT

In [29]:
%%writefile dataset_summarizer.py
"""Resume un DataFrame y exporta el resumen a un archivo TXT."""

from pathlib import Path
import pandas as pd


class DatasetSummarizer:
    """Crea un resumen de dimensiones, tipos, nulos y estadísticas."""

    def __init__(self, dataframe: pd.DataFrame):
        self.dataframe = dataframe

    def build_summary(self) -> str:
        """Construye y devuelve el resumen como texto."""
        df = self.dataframe
        lines = [
            "RESUMEN DEL DATASET",
            "=" * 60,
            f"Cantidad de filas: {df.shape[0]}",
            f"Cantidad de columnas: {df.shape[1]}",
            "",
            "COLUMNAS Y TIPOS DE DATOS",
            "-" * 60,
        ]
        if df.shape[1] == 0:
            lines.append("El DataFrame no tiene columnas.")
        else:
            for column in df.columns:
                lines.append(f"- {column}: {df[column].dtype}")

        lines.extend(["", "VALORES NULOS POR COLUMNA", "-" * 60])
        if df.shape[1] == 0:
            lines.append("No hay columnas para revisar.")
        else:
            for column, count in df.isna().sum().items():
                lines.append(f"- {column}: {int(count)}")

        lines.extend(["", "ESTADÍSTICAS DESCRIPTIVAS", "-" * 60])
        lines.append(
            df.describe(include="all").to_string()
            if not df.empty else "El DataFrame está vacío."
        )
        lines.append("")
        return "\n".join(lines)

    def save_summary(self, output_path: str = "resumen_dataset.txt") -> str:
        """Guarda el resumen en UTF-8 y devuelve la ruta del archivo."""
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(self.build_summary(), encoding="utf-8")
        return str(path)


Overwriting dataset_summarizer.py


## 3. Importar las clases y cargar el dataset
Se carga el split `train` del dataset público `scikit-learn/iris` desde Hugging Face.

In [30]:
import pandas as pd
from IPython.display import display
from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

dataset_id = "scikit-learn/iris"
loader = HFDatasetLoader(dataset_id=dataset_id, split="train")
df_original = loader.load_dataframe()

print("Dataset:", dataset_id)
print("Dimensiones originales:", df_original.shape)
print("Columnas originales:", df_original.columns.tolist())
display(df_original.head())

Dataset: scikit-learn/iris
Dimensiones originales: (150, 6)
Columnas originales: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 4. Transformación

In [31]:
df_t1 = df_original.rename(columns={"sepal_length": "longitud_sepalo"}).copy()
print("Transformación 1 — columnas después de renombrar:")
print(df_t1.columns.tolist())
display(df_t1.head())

Transformación 1 — columnas después de renombrar:
['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


##Transformación 1: renombrar columnas

In [32]:

# Transformación 1: renombrar columnas
df_t1 = df_original.copy()

df_t1 = df_t1.rename(columns={
    "SepalLengthCm": "longitud_sepalo",
    "SepalWidthCm": "ancho_sepalo",
    "PetalLengthCm": "petal_length",
    "PetalWidthCm": "petal_width",
    "Species": "target"
})

print("Transformación 1 — columnas renombradas:")
print(df_t1.columns.tolist())

display(df_t1.head())


Transformación 1 — columnas renombradas:
['Id', 'longitud_sepalo', 'ancho_sepalo', 'petal_length', 'petal_width', 'target']


,Id,longitud_sepalo,ancho_sepalo,petal_length,petal_width,target
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


# Transformación 2: seleccionar columnas

In [33]:

# Transformación 2: seleccionar columnas
columnas_seleccionadas = [
    "longitud_sepalo",
    "petal_length",
    "petal_width",
    "target"
]

df_t2 = df_t1[columnas_seleccionadas].copy()

print("Transformación 2 — dimensiones:", df_t2.shape)
display(df_t2.head())


Transformación 2 — dimensiones: (150, 4)


,longitud_sepalo,petal_length,petal_width,target
0,5.1,1.4,0.2,Iris-setosa
1,4.9,1.4,0.2,Iris-setosa
2,4.7,1.3,0.2,Iris-setosa
3,4.6,1.5,0.2,Iris-setosa
4,5.0,1.4,0.2,Iris-setosa


# Transformación 3: filtrar flores con pétalos de más de 3 cm

In [34]:

# Transformación 3: filtrar flores con pétalos de más de 3 cm
df_t3 = df_t2[df_t2["petal_length"] > 3.0].copy()

print("Transformación 3 — dimensiones:", df_t3.shape)
display(df_t3.head())


Transformación 3 — dimensiones: (99, 4)


,longitud_sepalo,petal_length,petal_width,target
50,7.0,4.7,1.4,Iris-versicolor
51,6.4,4.5,1.5,Iris-versicolor
52,6.9,4.9,1.5,Iris-versicolor
53,5.5,4.0,1.3,Iris-versicolor
54,6.5,4.6,1.5,Iris-versicolor


# Transformación 4: crear una columna calculada

In [35]:

# Transformación 4: crear una columna calculada
df_transformado = df_t3.copy()

df_transformado["area_petalo_estimada"] = (
    df_transformado["petal_length"] *
    df_transformado["petal_width"]
)

print("Transformación 4 — dimensiones:", df_transformado.shape)
display(df_transformado.head())


Transformación 4 — dimensiones: (99, 5)


,longitud_sepalo,petal_length,petal_width,target,area_petalo_estimada
50,7.0,4.7,1.4,Iris-versicolor,6.58
51,6.4,4.5,1.5,Iris-versicolor,6.75
52,6.9,4.9,1.5,Iris-versicolor,7.35
53,5.5,4.0,1.3,Iris-versicolor,5.20
54,6.5,4.6,1.5,Iris-versicolor,6.90


## 8. Generar el resumen TXT desde el DataFrame final
Importante: se entrega `df_transformado` a la clase de resumen, por lo que el `.txt` corresponde a los datos después de las cuatro transformaciones.

In [36]:

from dataset_summarizer import DatasetSummarizer

resumidor = DatasetSummarizer(df_transformado)
resumidor.save_summary("resumen_dataset.txt")

print("Resumen generado correctamente.")
print("Columnas finales:", df_transformado.columns.tolist())


Resumen generado correctamente.
Columnas finales: ['longitud_sepalo', 'petal_length', 'petal_width', 'target', 'area_petalo_estimada']


## 9. Descargar el TXT generado

In [37]:
from google.colab import files
files.download("resumen_dataset.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Explicación de las transformaciones realizadas

Explicación de las transformaciones realizadas

Transformación 1. Renombrar columnas: Se cambiaron los nombres de algunas columnas del dataset Iris para facilitar su comprensión y mantener nombres consistentes durante el análisis.

Transformación 2. Seleccionar columnas: Se seleccionaron cuatro columnas relevantes: longitud_sepalo, petal_length, petal_width y target, conservando los 150 registros originales.

Transformación 3. Filtrar registros: Se seleccionaron únicamente las flores cuyo valor de petal_length es mayor que 3.0 cm. Como resultado, se obtuvieron 99 registros.

Transformación 4. Crear una nueva columna: Se agregó la columna area_petalo_estimada, calculada multiplicando petal_length por petal_width. Esta variable representa una estimación sencilla basada en las dimensiones del pétalo.

Finalmente, se generó el archivo resumen_dataset.txt, que contiene las dimensiones del DataFrame transformado, los tipos de datos, los valores nulos y las estadísticas descriptivas de las columnas.